# Modelado - Enfoque B: TimeSeries Cross-Validation
En esta notebook utilizaremos la validación cruzada para series temporales (`TimeSeriesSplit`). Evaluaremos el modelo sobre el 85% de los datos de desarrollo, respetando siempre el orden cronológico en cada partición.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate
import matplotlib.pyplot as plt

### 1. Carga de Datos

In [ ]:
base_path = '../data/model_input/cv/'

X_train_full = pd.read_csv(base_path + 'X_train_full.csv')
y_train_full = pd.read_csv(base_path + 'y_train_full.csv').values.ravel()

print(f"X_train_full shape (85% de los datos): {X_train_full.shape}")

### 2. Definición del Preprocesador (ColumnTransformer)

In [ ]:
vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = [
    'is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

### 3. Configuración del Pipeline y Cross-Validation

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_model)
])

pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', lr_model)
])


# Configuramos 5 particiones temporales
tscv = TimeSeriesSplit(n_splits=5)

### 4. Ejecución y Evaluación

In [ ]:
def ejecutar_cv(pipeline, nombre_modelo):
    print(f"\n{'='*40}")
    print(f"Ejecutando CV para: {nombre_modelo}")
    print(f"{'='*40}")
    
    cv_results = cross_validate(
        pipeline, 
        X_train_full, 
        y_train_full, 
        cv=tscv, 
        scoring=['roc_auc', 'f1', 'accuracy'],
        n_jobs=-1
    )
    
    for i in range(5):
        print(f"Fold {i+1} - ROC-AUC: {cv_results['test_roc_auc'][i]:.4f} | F1: {cv_results['test_f1'][i]:.4f} | Accuracy: {cv_results['test_accuracy'][i]:.4f}")
    
    print("\n--- Resumen Global ---")
    print(f"ROC-AUC Promedio: {cv_results['test_roc_auc'].mean():.4f} (+/- {cv_results['test_roc_auc'].std() * 2:.4f})")
    print(f"F1-Score Promedio: {cv_results['test_f1'].mean():.4f} (+/- {cv_results['test_f1'].std() * 2:.4f})")
    print(f"Accuracy Promedio: {cv_results['test_accuracy'].mean():.4f} (+/- {cv_results['test_accuracy'].std() * 2:.4f})")

# Ejecutar CV para ambos modelos
ejecutar_cv(pipeline_lr, "Regresión Logística")
ejecutar_cv(pipeline_rf, "Random Forest")



# Random Forest (static vs cv)

Aquí te detallo las 3 conclusiones más importantes que podemos sacar al comparar el enfoque estático con este CV:

### 1. El modelo es robusto y consistente
Fíjate en los promedios globales del CV:
*   **ROC-AUC Promedio:** 0.8323 (vs. 0.8457 en el estático)
*   **F1-Score Promedio:** 0.6388 (vs. 0.65 en el estático)

Los números son **muy similares**. Esto es una excelente noticia porque nos confirma que el buen resultado que vimos en la validación estática no fue "suerte" por haber cortado los datos en un punto favorable. El Random Forest realmente tiene esa capacidad predictiva de forma sostenida.

### 2. El "Misterio" del Fold 1
Si miras el detalle, el **Fold 1** tiene un rendimiento notablemente más bajo (ROC-AUC 0.74, F1 0.53). ¿Por qué pasa esto?
No es que el modelo sea malo, es por cómo funciona el `TimeSeriesSplit`:
*   En el Fold 1, el modelo se entrena con un pedacito muy pequeño de datos (el principio de la historia) y tiene que predecir el siguiente bloque. Al tener poca historia, le cuesta generalizar.
*   A medida que avanzamos a los **Folds 2, 3, 4 y 5**, el modelo va acumulando más meses de historia para entrenar. Fíjate cómo a partir del Fold 2 el rendimiento salta a ~0.84 y se mantiene súper estable hasta el final. ¡Esto demuestra que **mientras más datos históricos le damos, mejor y más estable se vuelve**!

### 3. La realidad del negocio (La varianza)
El `(+/- 0.0892)` en el ROC-AUC nos dice que el rendimiento del modelo puede fluctuar un poco dependiendo de la época del año. Esto tiene todo el sentido en la hotelería: predecir cancelaciones en temporada baja no es igual que en temporada alta. Sin embargo, la variación es pequeña (especialmente si quitamos el Fold 1), lo que nos da tranquilidad para llevarlo a producción.

---

### Veredicto Final:
**El Random Forest es el ganador indiscutible.** La validación cruzada temporal acaba de certificar que el modelo es estable, aprende bien a medida que acumula datos y no está sobreajustado (overfitted).